# 📦 Upload model weights from Google Drive → Zenodo

Este notebook sube los pesos del modelo directamente desde tu Google Drive a Zenodo **sin descargarlos a tu ordenador**.

## Pasos previos
1. Ve a [zenodo.org](https://zenodo.org) → tu cuenta → **Applications** → **Personal access tokens**
2. Crea un token con scopes: `deposit:write` y `deposit:actions`
3. Pégalo en la celda de configuración de abajo (o mejor, añádelo como **Colab Secret** con nombre `ZENODO_TOKEN`)

In [ ]:
# ── Celda 1: Montar Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Celda 2: Configuración ─────────────────────────────────────────────────────
import os

# --- TOKEN ---
# Opción A (recomendada): usar Colab Secrets (icono 🔑 en el panel izquierdo)
try:
    from google.colab import userdata
    ZENODO_TOKEN = userdata.get('ZENODO_TOKEN')
    print("✅ Token cargado desde Colab Secrets")
except Exception:
    # Opción B: pégalo aquí directamente (no lo subas a GitHub nunca)
    ZENODO_TOKEN = "PEGA_TU_TOKEN_AQUI"
    print("⚠️  Token cargado desde código — recuerda no subir esto a GitHub")

# --- RUTA BASE EN DRIVE ---
# Ajusta esta ruta a donde estén tus archivos
DRIVE_BASE = "/content/drive/MyDrive/rgi_Klebsiella_pneumoniae_ertapenem"

# --- ARCHIVOS A SUBIR ---
# Edita esta lista si algún archivo tiene nombre diferente en tu Drive
FILES_TO_UPLOAD = [
    # Backbone principal (usado en notebooks 04 y 05)
    f"{DRIVE_BASE}/modelo_MIL_v5.1_best_auc.pth",
    # XGBoost final (usado en notebooks 04 y 05)
    f"{DRIVE_BASE}/modelo_xgboost_v5.3.pkl",
    # Los 5 folds MIL (usados en notebook 03)
    f"{DRIVE_BASE}/cv_folds/fold1_modelo_best_auc.pth",
    f"{DRIVE_BASE}/cv_folds/fold2_modelo_best_auc.pth",
    f"{DRIVE_BASE}/cv_folds/fold3_modelo_best_auc.pth",
    f"{DRIVE_BASE}/cv_folds/fold4_modelo_best_auc.pth",
    f"{DRIVE_BASE}/cv_folds/fold5_modelo_best_auc.pth",
    # Los 5 folds XGBoost (usados en notebook 03)
    f"{DRIVE_BASE}/cv_folds/fold1_xgboost.pkl",
    f"{DRIVE_BASE}/cv_folds/fold2_xgboost.pkl",
    f"{DRIVE_BASE}/cv_folds/fold3_xgboost.pkl",
    f"{DRIVE_BASE}/cv_folds/fold4_xgboost.pkl",
    f"{DRIVE_BASE}/cv_folds/fold5_xgboost.pkl",
]

# Verificar que todos los archivos existen antes de empezar
missing = [f for f in FILES_TO_UPLOAD if not os.path.exists(f)]
if missing:
    print("❌ Archivos NO encontrados en Drive:")
    for m in missing:
        print(f"   {m}")
    print("\n⚠️  Edita DRIVE_BASE o FILES_TO_UPLOAD antes de continuar.")
else:
    sizes = {f: os.path.getsize(f) / 1e9 for f in FILES_TO_UPLOAD}
    total_gb = sum(sizes.values())
    print(f"✅ Todos los archivos encontrados. Tamaño total: {total_gb:.2f} GB")
    for f, s in sizes.items():
        print(f"   {os.path.basename(f):45s}  {s:.2f} GB")

In [ ]:
# ── Celda 3: Crear borrador en Zenodo ─────────────────────────────────────────
import requests, json

ZENODO_API = "https://zenodo.org/api"
HEADERS = {"Content-Type": "application/json"}
PARAMS  = {"access_token": ZENODO_TOKEN}

# Metadatos del depósito
metadata = {
    "metadata": {
        "title": "Trained model weights — Lineage-aware genomic learning exposes clonal inflation in ertapenem resistance prediction",
        "upload_type": "dataset",
        "description": (
            "Trained PyTorch checkpoints (NTv3MultiHeadMIL) and XGBoost models "
            "for ertapenem resistance prediction in Klebsiella pneumoniae using "
            "ST-blocked cross-validation. Includes 5-fold MIL checkpoints, "
            "5-fold XGBoost models, the pretrained MIL v5.1 backbone, and the "
            "final XGBoost v5.3 classifier. Code: https://github.com/TODO/TODO"
        ),
        "access_right": "open",
        "license": "mit-license",
        # TODO: añade tus autores aquí
        "creators": [
            {"name": "TODO Apellido, Nombre", "affiliation": "TODO afiliación"}
        ],
        "keywords": [
            "antimicrobial resistance", "Klebsiella pneumoniae", "ertapenem",
            "multiple instance learning", "Nucleotide Transformer", "XGBoost",
            "ST-blocked cross-validation", "genomics"
        ]
    }
}

r = requests.post(f"{ZENODO_API}/depositions", params=PARAMS,
                  json=metadata, headers=HEADERS)

if r.status_code != 201:
    print(f"❌ Error creando depósito: {r.status_code}")
    print(r.json())
else:
    deposition = r.json()
    DEPOSITION_ID = deposition["id"]
    BUCKET_URL    = deposition["links"]["bucket"]
    DEPOSIT_URL   = deposition["links"]["html"]
    print(f"✅ Borrador creado en Zenodo")
    print(f"   Deposition ID : {DEPOSITION_ID}")
    print(f"   Ver borrador  : {DEPOSIT_URL}")
    print(f"   Bucket URL    : {BUCKET_URL}")

In [ ]:
# ── Celda 4: Subir archivos (streamed, sin cargar en RAM) ─────────────────────
import time

def upload_file(bucket_url, filepath, token, chunk_size=8*1024*1024):
    """Sube un archivo a Zenodo en streaming (no lo carga entero en RAM)."""
    filename = os.path.basename(filepath)
    file_size = os.path.getsize(filepath)
    print(f"\n⬆️  Subiendo: {filename}  ({file_size/1e9:.2f} GB)")

    with open(filepath, "rb") as fp:
        r = requests.put(
            f"{bucket_url}/{filename}",
            data=fp,
            params={"access_token": token},
            stream=True,
            timeout=7200  # 2 horas por archivo
        )

    if r.status_code in (200, 201):
        checksum = r.json().get("checksum", "N/A")
        print(f"   ✅ OK  |  checksum: {checksum}")
        return True
    else:
        print(f"   ❌ Error {r.status_code}: {r.text[:300]}")
        return False

results = {}
for fpath in FILES_TO_UPLOAD:
    if not os.path.exists(fpath):
        print(f"⏭️  Saltando (no existe): {fpath}")
        results[fpath] = "MISSING"
        continue
    ok = upload_file(BUCKET_URL, fpath, ZENODO_TOKEN)
    results[fpath] = "OK" if ok else "ERROR"

print("\n" + "="*60)
print("RESUMEN FINAL")
print("="*60)
for f, s in results.items():
    icon = "✅" if s == "OK" else ("⏭️" if s == "MISSING" else "❌")
    print(f"{icon}  {os.path.basename(f):45s}  {s}")

In [ ]:
# ── Celda 5: (OPCIONAL) Publicar el depósito ──────────────────────────────────
# ⚠️  ATENCIÓN: una vez publicado NO se puede borrar, solo versionar.
# Revisa primero el borrador en el enlace de arriba antes de ejecutar esto.

PUBLICAR = False  # Cambia a True cuando estés seguro

if PUBLICAR:
    r = requests.post(
        f"{ZENODO_API}/depositions/{DEPOSITION_ID}/actions/publish",
        params={"access_token": ZENODO_TOKEN}
    )
    if r.status_code == 202:
        doi = r.json()["doi"]
        url = r.json()["links"]["html"]
        print(f"🎉 Publicado correctamente!")
        print(f"   DOI : {doi}")
        print(f"   URL : {url}")
        print(f"\n👉 Pon este DOI en README.md y CITATION.cff del repo GitHub")
    else:
        print(f"❌ Error publicando: {r.status_code}")
        print(r.json())
else:
    print("ℹ️  Modo borrador. Cambia PUBLICAR=True cuando hayas revisado el depósito.")
    print(f"   Revisa aquí: {DEPOSIT_URL}")